# Micrograd research - on automatic gradient engine e.g micrograd from karpathy/geohot

In [1]:
# what's the goal of derivative 
# why neural network uses derivative
# how neural network learns using backpropagation and gradient decent

The Big Idea of machine learning / core of any point of a neural nets is this:

given a function $f(x)$, we want to make f smaller/larger - we can only change the value of x, so how much of x should be changed, which direction should we nudge x, and by how much

In [5]:
import math
import random

# ============================================================
# 1. THE AUTOGRAD ENGINE
# ============================================================

class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    def __pow__(self, n):
        assert isinstance(n, (int, float))
        out = Value(self.data ** n, (self,), f'**{n}')
        def _backward():
            self.grad += n * (self.data ** (n - 1)) * out.grad
        out._backward = _backward
        return out

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __truediv__(self, other): return self * (other ** -1)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __rsub__(self, other): return other + (-self)
    def __rtruediv__(self, other): return other * (self ** -1)

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        e = math.exp(self.data)
        out = Value(e, (self,), 'exp')
        def _backward():
            self.grad += e * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), 'relu')
        def _backward():
            self.grad += (1.0 if self.data > 0 else 0.0) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()


# ============================================================
# 2. NEURAL NET BUILDING BLOCKS
# ============================================================

class Neuron:
    def __init__(self, nin):
        # one weight per input, plus a bias
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)

    def __call__(self, x):
        # x is a list of Values (one sample, nin features)
        act = self.b
        for xi, wi in zip(x, self.w):
            act = act + xi * wi
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP:
    def __init__(self, nin, nouts):
        sizes = [nin] + nouts
        self.layers = [Layer(sizes[i], sizes[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


# ============================================================
# 3. DUMMY DATASET: XOR
# ============================================================
# XOR is the classic "linear models can't do this" test.
# A single neuron fails. A 2-layer MLP nails it.

xs = [
    [Value(0.0), Value(0.0)],
    [Value(0.0), Value(1.0)],
    [Value(1.0), Value(0.0)],
    [Value(1.0), Value(1.0)],
]
ys = [0.0, 1.0, 1.0, 0.0]  # XOR targets


# ============================================================
# 4. TRAINING LOOP
# ============================================================

random.seed(42)

# 2 inputs -> hidden layer of 4 tanh neurons -> 1 output tanh neuron
model = MLP(2, [4, 1])
params = model.parameters()
print(f"Total parameters: {len(params)}\n")

learning_rate = 0.1
epochs = 200

for epoch in range(epochs):
    # ---------- forward ----------
    preds = [model(x) for x in xs]
    # MSE loss: mean over samples of (pred - target)^2
    loss = sum((p + (-y)) ** 2 for p, y in zip(preds, ys)) * (1.0 / len(ys))

    # ---------- backward ----------
    # zero grads first! (they accumulate via +=)
    for p in params:
        p.grad = 0.0
    loss.backward()

    # ---------- update ----------
    for p in params:
        p.data -= learning_rate * p.grad

    # ---------- log ----------
    if epoch % 20 == 0 or epoch == epochs - 1:
        print(f"epoch {epoch:3d}  loss = {loss.data:.6f}")


# ============================================================
# 5. RESULTS
# ============================================================

print("\n--- Final predictions ---")
for x, y in zip(xs, ys):
    pred = model(x)
    print(f"input=({x[0].data:.0f},{x[1].data:.0f})  "
          f"target={y:.0f}  pred={pred.data:.4f}")

Total parameters: 17

epoch   0  loss = 0.335945
epoch  20  loss = 0.214293
epoch  40  loss = 0.199819
epoch  60  loss = 0.185309
epoch  80  loss = 0.170131
epoch 100  loss = 0.153768
epoch 120  loss = 0.135957
epoch 140  loss = 0.117062
epoch 160  loss = 0.098161
epoch 180  loss = 0.080598
epoch 199  loss = 0.066047

--- Final predictions ---
input=(0,0)  target=0  pred=0.1064
input=(0,1)  target=1  pred=0.6638
input=(1,0)  target=1  pred=0.7149
input=(1,1)  target=0  pred=0.2361


In [ ]:
import math
import random

# ============================================================
# 1. AUTOGRAD ENGINE
# ============================================================

class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad  += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _backward
        return out

    def __pow__(self, n):
        assert isinstance(n, (int, float))
        out = Value(self.data ** n, (self,), f'**{n}')
        def _backward():
            self.grad += n * (self.data ** (n - 1)) * out.grad
        out._backward = _backward
        return out

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __truediv__(self, other): return self * (other ** -1)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __rsub__(self, other): return other + (-self)
    def __rtruediv__(self, other): return other * (self ** -1)

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def exp(self):
        e = math.exp(self.data)
        out = Value(e, (self,), 'exp')
        def _backward():
            self.grad += e * out.grad
        out._backward = _backward
        return out

    def log(self):
        d = max(self.data, 1e-12)   # clamp to avoid log(0)
        out = Value(math.log(d), (self,), 'log')
        def _backward():
            self.grad += (1.0 / d) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), 'relu')
        def _backward():
            self.grad += (1.0 if self.data > 0 else 0.0) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()


# ============================================================
# 2. NN BUILDING BLOCKS
# ============================================================

class Neuron:
    def __init__(self, nin, nonlin=True):
        scale = 1.0 / math.sqrt(nin)
        self.w = [Value(random.uniform(-1, 1) * scale) for _ in range(nin)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self, x):
        act = self.b
        for xi, wi in zip(x, self.w):
            act = act + xi * wi
        return act.tanh() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, nin, nout, **kwargs):
        self.neurons = [Neuron(nin, **kwargs) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP:
    def __init__(self, nin, nouts):
        sizes = [nin] + nouts
        self.layers = []
        for i in range(len(nouts)):
            is_last = (i == len(nouts) - 1)
            self.layers.append(Layer(sizes[i], sizes[i+1], nonlin=not is_last))

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x   # list of logits (linear output on last layer)

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


# ============================================================
# 3. SOFTMAX CROSS-ENTROPY LOSS
# ============================================================

def softmax_cross_entropy(logits, target_idx):
    """
    logits: list of Value, length C
    target_idx: int
    returns: scalar Value loss
    """
    max_val = max(z.data for z in logits)
    shifted = [z - max_val for z in logits]
    exps = [s.exp() for s in shifted]
    total = exps[0]
    for e in exps[1:]:
        total = total + e
    probs = [e / total for e in exps]
    return -probs[target_idx].log()


# ============================================================
# 4. SYNTHETIC 8x8 "DIGIT" DATASET
# ============================================================
# We render crude 8x8 templates for digits 0-4 and add noise.
# Small enough to train in seconds, non-trivial enough to learn.

IMG_SIZE = 8

def make_digit_image(digit, size=IMG_SIZE):
    """Return an size x size list of 0.0/1.0 pixels for digits 0-4."""
    img = [[0.0] * size for _ in range(size)]
    s = size

    if digit == 0:
        for i in range(1, s - 1):
            img[i][1] = 1.0
            img[i][s - 2] = 1.0
        for j in range(1, s - 1):
            img[1][j] = 1.0
            img[s - 2][j] = 1.0

    elif digit == 1:
        for i in range(s):
            img[i][s // 2] = 1.0

    elif digit == 2:
        for j in range(1, s - 1):
            img[1][j] = 1.0
            img[s // 2][j] = 1.0
            img[s - 2][j] = 1.0
        for i in range(1, s // 2):
            img[i][s - 2] = 1.0
        for i in range(s // 2, s - 1):
            img[i][1] = 1.0

    elif digit == 3:
        for j in range(1, s - 1):
            img[1][j] = 1.0
            img[s // 2][j] = 1.0
            img[s - 2][j] = 1.0
        for i in range(1, s - 1):
            img[i][s - 2] = 1.0

    elif digit == 4:
        for i in range(1, s // 2 + 1):
            img[i][1] = 1.0
        for j in range(1, s - 1):
            img[s // 2][j] = 1.0
        for i in range(1, s - 1):
            img[i][s - 2] = 1.0

    return img


def add_noise(img, p=0.1):
    """Flip each pixel with probability p."""
    return [[pix if random.random() > p else 1.0 - pix for pix in row]
            for row in img]


def flatten(img):
    """Return a flat list of Value, one per pixel."""
    return [Value(pix) for row in img for pix in row]


def print_image(img):
    for row in img:
        print("".join("█" if p > 0.5 else "·" for p in row))


# Build the dataset
NUM_CLASSES = 5
SAMPLES_PER_CLASS = 40

random.seed(0)
dataset = []
for digit in range(NUM_CLASSES):
    for _ in range(SAMPLES_PER_CLASS):
        img = add_noise(make_digit_image(digit), p=0.1)
        dataset.append((img, digit))

random.shuffle(dataset)
split = int(0.8 * len(dataset))
train_data = dataset[:split]
test_data  = dataset[split:]

print(f"Train samples: {len(train_data)}")
print(f"Test  samples: {len(test_data)}")
print(f"Classes:       {NUM_CLASSES}  (digits 0-{NUM_CLASSES-1})")
print(f"Image size:    {IMG_SIZE}x{IMG_SIZE}\n")

print("Example (digit 2, with noise):")
print_image(make_digit_image(2))
print()


# ============================================================
# 5. MODEL
# ============================================================
# 64 inputs (8x8 pixels) -> hidden layers -> 5 logits

INPUT_DIM  = IMG_SIZE * IMG_SIZE   # 64
HIDDEN     = [32, 32]
OUTPUT_DIM = NUM_CLASSES

model = MLP(INPUT_DIM, HIDDEN + [OUTPUT_DIM])
params = model.parameters()
print(f"Model: {INPUT_DIM} -> {HIDDEN} -> {OUTPUT_DIM}")
print(f"Parameters: {len(params)}\n")


# ============================================================
# 6. EVALUATION
# ============================================================

def evaluate(data):
    total_loss = 0.0
    correct = 0
    for img, label in data:
        x = flatten(img)
        logits = model(x)
        loss = softmax_cross_entropy(logits, label)
        total_loss += loss.data
        pred = max(range(len(logits)), key=lambda i: logits[i].data)
        if pred == label:
            correct += 1
    return total_loss / len(data), correct / len(data)


# ============================================================
# 7. TRAINING LOOP (minibatch SGD)
# ============================================================

BATCH_SIZE = 16
LEARNING_RATE = 0.05
EPOCHS = 40

for epoch in range(EPOCHS):
    random.shuffle(train_data)
    epoch_loss = 0.0
    num_batches = len(train_data) // BATCH_SIZE

    for b in range(num_batches):
        batch = train_data[b * BATCH_SIZE : (b + 1) * BATCH_SIZE]

        # ---- forward: accumulate loss over batch ----
        batch_loss = Value(0.0)
        for img, label in batch:
            x = flatten(img)
            logits = model(x)
            batch_loss = batch_loss + softmax_cross_entropy(logits, label)
        batch_loss = batch_loss * (1.0 / BATCH_SIZE)

        # ---- backward ----
        for p in params:
            p.grad = 0.0
        batch_loss.backward()

        # ---- update ----
        for p in params:
            p.data -= LEARNING_RATE * p.grad

        epoch_loss += batch_loss.data

    if epoch % 2 == 0 or epoch == EPOCHS - 1:
        tr_loss, tr_acc = evaluate(train_data)
        te_loss, te_acc = evaluate(test_data)
        print(f"epoch {epoch:3d}  "
              f"train_loss={tr_loss:.4f}  train_acc={tr_acc:.3f}  "
              f"test_loss={te_loss:.4f}  test_acc={te_acc:.3f}")


# ============================================================
# 8. SHOW PREDICTIONS ON TEST SET
# ============================================================

print("\n--- Sample predictions ---")
for img, label in test_data[:10]:
    logits = model(flatten(img))
    pred = max(range(len(logits)), key=lambda i: logits[i].data)
    mark = "✓" if pred == label else "✗"
    print(f"true={label}  pred={pred}  {mark}  "
          f"probs={[round(math.exp(l.data) if False else l.data, 2) for l in logits]}")


# ============================================================
# 9. CONFUSION MATRIX
# ============================================================

print("\n--- Confusion matrix (test set) ---")
cm = [[0] * NUM_CLASSES for _ in range(NUM_CLASSES)]
for img, label in test_data:
    logits = model(flatten(img))
    pred = max(range(len(logits)), key=lambda i: logits[i].data)
    cm[label][pred] += 1

print("      " + "  ".join(f"p{c}" for c in range(NUM_CLASSES)))
for i in range(NUM_CLASSES):
    row = "  ".join(f"{v:2d}" for v in cm[i])
    print(f"t{i}    {row}")

Train samples: 160
Test  samples: 40
Classes:       5  (digits 0-4)
Image size:    8x8

Example (digit 2, with noise):
········
·██████·
······█·
······█·
·██████·
·█······
·██████·
········

Model: 64 -> [32, 32] -> 5
Parameters: 3301

epoch   0  train_loss=1.5459  train_acc=0.463  test_loss=1.5515  test_acc=0.400
epoch   2  train_loss=1.3644  train_acc=0.681  test_loss=1.3920  test_acc=0.700
epoch   4  train_loss=1.1138  train_acc=0.781  test_loss=1.1599  test_acc=0.750
epoch   6  train_loss=0.8691  train_acc=0.812  test_loss=0.9060  test_acc=0.825
epoch   8  train_loss=0.6819  train_acc=0.819  test_loss=0.7056  test_acc=0.850
epoch  10  train_loss=0.5417  train_acc=0.844  test_loss=0.5670  test_acc=0.850
